In [1]:
import transformers
import huggingface_hub
import tokenizers
print(transformers.__version__)
print(huggingface_hub.__version__)
print(tokenizers.__version__)

4.57.6
0.36.2
0.22.2


RuBERT — классификация и извлечение сущностей
- Модель: DeepPavlov/rubert-base-cased
- Подходит как базовая русскоязычная encoder-модель
- Для конкретной задачи обычно требуется подходящая выходная голова и, при необходимости, дообучение.

RuBERT-tiny2 — компактная модель
- Модель: cointegrated/rubert-tiny2
- Небольшая модель, удобная для экспериментов, извлечения эмбеддингов и дообучения.
- Не является универсальным готовым классификатором.

Russian T5 — генерация и преобразование текста
- Модель: ai-forever/ruT5-base
- Семейство encoder-decoder моделей для задач преобразования текста.
- Конкретный сценарий зависит от обученной версии.

ruGPT — генерация русского текста
- Семейство: ai-forever/ruGPT-3.5-13B и другие версии ruGPT
- Используется для продолжения текста и генерации.

Multilingual E5 — семантический поиск
- Модель: intfloat/multilingual-e5-small
- Получает векторы русских предложений и документов. Подходит для поиска похожих текстов и построения основы RAG.

# 1. Классификация текста и анализ тональности

In [2]:
from transformers import pipeline

model_name = "cointegrated/rubert-tiny-sentiment-balanced"

classifier = pipeline("text-classification", model=model_name)

reviews = [
    "Отличный телефон, работает быстро!",
    "Ужасное качество, деньги потрачены зря.",
    "Обычный товар, ничего особенного.",
    "Доставка быстрая, упаковка хорошая."
]

results = classifier(reviews)

for text, result in zip(reviews, results):
    print(f"Текст: {text}")
    print(f"Класс: {result['label']}")
    print(f"Оценка: {result['score']:.3f}")
    print("-" * 50)

Device set to use cpu


Текст: Отличный телефон, работает быстро!
Класс: positive
Оценка: 0.997
--------------------------------------------------
Текст: Ужасное качество, деньги потрачены зря.
Класс: negative
Оценка: 0.960
--------------------------------------------------
Текст: Обычный товар, ничего особенного.
Класс: neutral
Оценка: 0.452
--------------------------------------------------
Текст: Доставка быстрая, упаковка хорошая.
Класс: positive
Оценка: 0.974
--------------------------------------------------


# 2. Zero-shot classification

In [4]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="cointegrated/rubert-base-cased-nli-threeway")

categories = [
    "Проблема с оплатой",
    "Техническая проблема",
    "Доставка заказа",
    "Возврат товара"
]

texts = [
    "Не могу оплатить заказ банковской картой.",
    "Приложение постоянно зависает.",
    "Курьер не приехал в назначенное время.",
    "Хочу вернуть товар и получить деньги."
]

for text in texts:
    result = classifier(
        text,
        candidate_labels=categories
    )

    print("\nОбращение:", text)

    for label, score in zip(
        result["labels"],
        result["scores"]
    ):
        print(f"{label}: {score:.3f}")

config.json: 0.00B [00:00, ?B/s]

C:\Users\D\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\D\.cache\huggingface\hub\models--cointegrated--rubert-base-cased-nli-threeway. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu



Обращение: Не могу оплатить заказ банковской картой.
Проблема с оплатой: 0.705
Доставка заказа: 0.142
Возврат товара: 0.082
Техническая проблема: 0.070

Обращение: Приложение постоянно зависает.
Техническая проблема: 0.383
Проблема с оплатой: 0.371
Возврат товара: 0.162
Доставка заказа: 0.085

Обращение: Курьер не приехал в назначенное время.
Доставка заказа: 0.347
Проблема с оплатой: 0.270
Возврат товара: 0.208
Техническая проблема: 0.175

Обращение: Хочу вернуть товар и получить деньги.
Возврат товара: 0.788
Проблема с оплатой: 0.102
Доставка заказа: 0.101
Техническая проблема: 0.009


# 3. Извлечение именованных сущностей — NER

In [5]:
from transformers import pipeline

ner = pipeline("token-classification", model="r1char9/ner-rubert-tiny-news", aggregation_strategy="simple")

text = (
    "Сбербанк открыл новый офис в Москве. "
    "Об этом сообщил Андрей Иванов."
    "Публикация произошла в газете Известия."
)

entities = ner(text)

for entity in entities:
    print({
        "text": entity["word"],
        "type": entity["entity_group"],
        "score": round(entity["score"], 3)
    })

Device set to use cpu


{'text': 'Сбербанк', 'type': 'ORG', 'score': np.float32(0.979)}
{'text': 'Москве', 'type': 'LOC', 'score': np.float32(0.984)}
{'text': 'Андрей Иванов.', 'type': 'PER', 'score': np.float32(0.872)}
{'text': 'газете', 'type': 'MEDIA', 'score': np.float32(0.859)}
{'text': 'Известия.', 'type': 'MEDIA', 'score': np.float32(0.914)}


# 4. Ответы на вопросы по контексту

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "r1char9/ruT5_q_a"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

context = """
Компания Альфа разрабатывает программное обеспечение.
Главный офис компании находится в Москве.
В 2025 году компания запустила новый сервис
для автоматизации обработки документов.
"""

questions = ["Где находится главный офис компании?", "Чем занимается компания Альфа?", "В каком году запущен сервис обработки документов?"]

for question in questions:
    inputs = tokenizer(
        context,
        question,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    
    model.eval()
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=3
        )
    
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Вопрос:", question)
    print("Ответ:", answer)

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
C:\Users\D\anaconda3\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Вопрос: Где находится главный офис компании?
Ответ: в Москве
Вопрос: Чем занимается компания Альфа?
Ответ: разрабатывает программное обеспечение
Вопрос: В каком году запущен сервис обработки документов?
Ответ: В 2025 году


# 5. Суммаризация текста

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "cointegrated/rut5-base-absum"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = """
Компания провела исследование качества клиентского сервиса.
Выяснилось, что большинство клиентов удовлетворены скоростью обработки обращений. 
Однако пользователи часто жалуются на отсутствие информации о статусе заказа. 
Компания планирует внедрить автоматические уведомления, улучшить работу службы поддержки и сократить время ожидания ответа.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

model.eval()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=3,
        do_sample=False
    )

summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Исходный текст:\n", text)
print("\nКраткое содержание:\n", summary)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Исходный текст:
 
Компания провела исследование качества клиентского сервиса.
Выяснилось, что большинство клиентов удовлетворены скоростью обработки обращений. 
Однако пользователи часто жалуются на отсутствие информации о статусе заказа. 
Компания планирует внедрить автоматические уведомления, улучшить работу службы поддержки и сократить время ожидания ответа.


Краткое содержание:
 По данным компании, клиенты удовлетворены скоростью обработки обращений.


# 6. Генерация текста на русском языке

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

messages = [
    {
        "role": "system",
        "content": "Ты преподаватель машинного обучения."
    },
    {
        "role": "user",
        "content": (
            "Объясни разницу между классификацией "
            "и регрессией. Приведи пример на Python."
        )
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
)

model.eval()

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=200,
        do_sample=False
    )

answer_tokens = outputs[0, inputs.shape[-1]:]

answer = tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)

print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Классификация и регрессия - два основных раздела в машинном обучении, которые используются для решения различных задач. Вот их ключевые различия:

1. **Классификация**:
   - Основная задача: определение классов или групп данных.
   - Пример: Построение модели для классификации текстовых сообщений (например, классификатор для обнаружения рекламных кампаний).
   - Входные данные: текстовые данные.
   - Выходные данные: классы (например, "да", "нет").

2. **Регрессия**:
   - Основная задача: прогнозирование значений.
   - Пример: Построение модели для прогнозирования цены товара.
   - Входные данные: набор данных, включая временные данные.
   - Выходные данные: значения, которые
